## 5. Merge Multi-Versión de Target + Preoperatorio

Este notebook une automáticamente el dataset preoperatorio con **todas** las versiones de target generadas (`OPERA_POS_*.xlsx`).


In [2]:
from pathlib import Path
import warnings
import pandas as pd

import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils.multi_version import run_merge_pipeline
from utils.version_config import ACTIVE_VERSIONS, VERSION_HISTORY, get_active_target_export_names

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data")
MERGED_DIR = DATA_DIR / "merged_versions"
MERGED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carga de Datasets Fuente

In [3]:
active_target_files = get_active_target_export_names()

print(f"Versiones activas: {ACTIVE_VERSIONS}")
display(pd.DataFrame(VERSION_HISTORY))

merge_artifacts = run_merge_pipeline(
    data_dir=DATA_DIR,
    merged_dir=MERGED_DIR,
    active_target_files=active_target_files,
)

print("Dataset Preoperatorio:")
print(f"  Ruta: {merge_artifacts['preop_path']}")
print(
    f"  Filas: {merge_artifacts['preop_shape'][0]:,} × "
    f"Columnas: {merge_artifacts['preop_shape'][1]}"
 )

Versiones activas: ['v1', 'v2', 'v3_sin_liquidos']


,version,threshold,flags_to_use,excluded_flags,apply_cancel_non_medico_rule,export_name,description,executed_on,notes
0,v1,1,RELEVANT_FLAGS,EXCLUDED_FLAGS,True,OPERA_POS_v1.xlsx,Sin agregación (>=1 flag),2026-02-28,Baseline histórico
1,v2,2,RELEVANT_FLAGS,EXCLUDED_FLAGS,True,OPERA_POS_v2.xlsx,Con agregación (>=2 flags),2026-02-28,Baseline histórico
2,v3_sin_liquidos,1,RELEVANT_FLAGS - flag_liquidos,EXCLUDED_FLAGS,True,OPERA_POS_v3_sin_liquidos.xlsx,Sin flag_liquidos (>=1 flag),2026-02-28,Sensibilidad clínica


Dataset Preoperatorio:
  Ruta: ../data/OPERA_PRE.xlsx
  Filas: 30,962 × Columnas: 236


In [4]:
print(f"Versiones target detectadas (activas): {len(merge_artifacts['target_files'])}")
for path in merge_artifacts["target_files"]:
    print(f"  - {Path(path).name}")

if merge_artifacts.get("ignored_target_files"):
    print("\nArchivos target ignorados (no activos):")
    for name in merge_artifacts["ignored_target_files"]:
        print(f"  - {name}")

Versiones target detectadas (activas): 3
  - OPERA_POS_v1.xlsx
  - OPERA_POS_v2.xlsx
  - OPERA_POS_v3_sin_liquidos.xlsx

Archivos target ignorados (no activos):
  - OPERA_POS.xlsx


## 2. Merge por Versión

In [5]:
merged_by_version = merge_artifacts["merged_by_version"]
df_merge_summary = merge_artifacts["merge_summary"]
df_merge_summary

,version,target_file,rows,cols,target_0,target_1,target_1_pct
0,v1,OPERA_POS_v1.xlsx,29865,237,15122,14743,49.37
1,v2,OPERA_POS_v2.xlsx,29865,237,23454,6411,21.47
2,v3_sin_liquidos,OPERA_POS_v3_sin_liquidos.xlsx,29865,237,15442,14423,48.29


In [6]:
first_version = df_merge_summary.iloc[0]["version"]
print(f"Primera versión detectada: {first_version}")
merged_by_version[first_version].head(3)

Primera versión detectada: v1


,Documento PMD,Edad,Peso (Kg),Talla (cm),IMC,Antecedentes anestésicos,Tensión Arterial Sistólica (mm/Hg),Tensión Arterial Diastólica (mm/Hg),Tensión Arterial Media (mm/Hg),Frecuencia Respiratoria,Temperatura,Puntaje Mallampati,anio,mes,dia_semana,Hora_decimal,Sexo_encoded,Atención_encoded,Grupo Sanguíneo_A,Grupo Sanguíneo_AB,Grupo Sanguíneo_B,Grupo Sanguíneo_O,Grupo Sanguíneo_Sin Dato,RH_Negativo,RH_Positivo,RH_Sin Dato,Antecedente neurológicos_convulsiones,Antecedente neurológicos_negativo,Antecedente neurológicos_neoplasia,Antecedente neurológicos_neuropatia,Antecedente neurológicos_tce,Antecedente neurológicos_trastorno psiquiatrico,Antecedente respiratorios_asma,Antecedente respiratorios_epoc,Antecedente respiratorios_negativo,Antecedente respiratorios_negativoneumonia,Antecedente respiratorios_neumonia,Antecedentes cardiovasculares_angina,Antecedentes cardiovasculares_arritmias,Antecedentes cardiovasculares_cardiopatias,Antecedentes cardiovasculares_hta,Antecedentes cardiovasculares_icc,Antecedentes cardiovasculares_infarto,Antecedentes cardiovasculares_negativo,Antecedentes cardiovasculares_valvulopatia,Antecedente hematológicos _anemia,Antecedente hematológicos _anticoagulantes,Antecedente hematológicos _aspirina,Antecedente hematológicos _discrasias,Antecedente hematológicos _negativo,Antecedente hematológicos _plavix,Antecedente hematológicos _transfusiones,Antecedente endocrinológicos_diabetes,Antecedente endocrinológicos_negativo,Antecedente endocrinológicos_obesidad,Antecedente endocrinológicos_tiroides,Antecedente renales_falla renal,Antecedente renales_infeccion urinaria,Antecedente renales_litiasis,Antecedente renales_negativo,Antecedente gastrointestinales_falla hepatica,Antecedente gastrointestinales_gastritis,Antecedente gastrointestinales_negativo,Antecedente gastrointestinales_rge,Anestesia previa_bag,Anestesia previa_bloqueo,Anestesia previa_conductiva,Anestesia previa_desconocido,Anestesia previa_epidural,Anestesia previa_espinal,Anestesia previa_general,Anestesia previa_intraarticular,Anestesia previa_local,Anestesia previa_neuroaxial,Anestesia previa_peribulbar,Anestesia previa_peridural,Anestesia previa_raquidea,Anestesia previa_regional,Anestesia previa_sedacion,Anestesia previa_topica,Examen_Hemoglobina(g/dl),Examen_PT (INR),Examen_Glicemia (mg/dl),Examen_Hematocrito (%),Examen_Plaquetas (ml),Examen_Leucocitos (ml),Examen_PTT,Examen_K (meq/L),Examen_Na (meq/L),Examen_P de O,Cuello Móvil_encoded,Apertura Oral_encoded,Estado Nutricional_encoded,Color de Piel_cianosis,Color de Piel_ictericia,Color de Piel_normal,Color de Piel_palidez,Sistema Nervioso_Sin dato,Sistema Nervioso_agitado,Sistema Nervioso_normal,Sistema Nervioso_par craneal anormal,Sistema Nervioso_sedado,Apertura Visual_encoded,Respuesta Verbal_encoded,Respuesta Motora_encoded,Sistema Respiratorio_auscultacion anormal,Sistema Respiratorio_disnea,Sistema Respiratorio_normal,Sistema Respiratorio_otro,Sistema cardiovascular_galope,Sistema cardiovascular_ingurgitacion yugular,Sistema cardiovascular_normal,Sistema cardiovascular_otro,Sistema cardiovascular_pulso anormal,Sistema cardiovascular_soplo,Abdomen_distension,Abdomen_masas,Abdomen_normal,Abdomen_otro,Arritmia_encoded,Disnea_encoded,Angina_encoded,Condición_alcohol,Condición_embarazo actual,Condición_farmacodependencia,Condición_fumador,Condición_hepatitis b,Condición_hipertermia maligna,Condición_ninguna,Condición_testigo jehova,Condición_vih,Tipo de anestesia propuesta_bloqueo iv,Tipo de anestesia propuesta_bloqueo n,Tipo de anestesia propuesta_general,Tipo de anestesia propuesta_local,Tipo de anestesia propuesta_peridural,Tipo de anestesia propuesta_raquidea,Tipo de anestesia propuesta_sedacion,Tipo de anestesia propuesta_sin dato,Prótesis Dental_fija,Prótesis Dental_movil,Prótesis Dental_no,Alérgeno_alimentos,Alérgeno_ambiental_animales,Alérgeno_contacto_hospitalario,Alérgeno_med_alergias_respiratorio,Alérgeno_med_analgesia_no_opioide,Alérgeno_med_anestesia_y_relajantes,Alérgeno_med_a

## 3. Resumen Comparativo entre Versiones

In [7]:
print("=" * 90)
print("RESUMEN MERGE PREOP + TARGET (MULTI-VERSIÓN)")
print("=" * 90)
print(df_merge_summary.to_string(index=False))

RESUMEN MERGE PREOP + TARGET (MULTI-VERSIÓN)
        version                    target_file  rows  cols  target_0  target_1  target_1_pct
             v1              OPERA_POS_v1.xlsx 29865   237     15122     14743         49.37
             v2              OPERA_POS_v2.xlsx 29865   237     23454      6411         21.47
v3_sin_liquidos OPERA_POS_v3_sin_liquidos.xlsx 29865   237     15442     14423         48.29


In [8]:
target_distribution = (
    df_merge_summary[["version", "target_0", "target_1", "target_1_pct"]]
    .sort_values("target_1_pct", ascending=False)
    .reset_index(drop=True)
)
target_distribution

,version,target_0,target_1,target_1_pct
0,v1,15122,14743,49.37
1,v3_sin_liquidos,15442,14423,48.29
2,v2,23454,6411,21.47


## 4. Exportación de Datasets Mergeados

In [9]:
df_exports = merge_artifacts["exports"]
df_exports

,version,path,rows
0,v1,../data/merged_versions/OPERA_COMPLETO_v1.xlsx,29865
1,v2,../data/merged_versions/OPERA_COMPLETO_v2.xlsx,29865
2,v3_sin_liquidos,../data/merged_versions/OPERA_COMPLETO_v3_sin_...,29865


## 5. Artefactos de Control para el Flujo Multi-Versión

In [10]:
print("=" * 90)
print("EXPORTACIÓN COMPLETADA")
print("=" * 90)
print(f"Resumen de merge: {merge_artifacts['summary_path']}")
print(f"Inventario de exports: {merge_artifacts['exports_path']}")
print("\nDatasets exportados:")
for _, row in df_exports.iterrows():
    print(f"  - [{row['version']}] {row['path']} ({row['rows']:,} filas)")

EXPORTACIÓN COMPLETADA
Resumen de merge: ../data/merged_versions/merge_summary_versions.csv
Inventario de exports: ../data/merged_versions/merged_exports_versions.csv

Datasets exportados:
  - [v1] ../data/merged_versions/OPERA_COMPLETO_v1.xlsx (29,865 filas)
  - [v2] ../data/merged_versions/OPERA_COMPLETO_v2.xlsx (29,865 filas)
  - [v3_sin_liquidos] ../data/merged_versions/OPERA_COMPLETO_v3_sin_liquidos.xlsx (29,865 filas)
